In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor,StackingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, ElasticNet,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings
from tqdm import tqdm
import os
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Concrete_Strength/")

In [10]:
concrete = pd.read_csv("Concrete_Data.csv")
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [11]:
X, y = concrete.drop("Strength", axis = 1), concrete["Strength"]
X.isna().sum()

Cement              0
Blast               0
Fly                 0
Water               0
Superplasticizer    0
Coarse              0
Fine                0
Age                 0
dtype: int64

In [12]:
ss = StandardScaler()
mm = MinMaxScaler()
kfold = KFold(n_splits=5,shuffle=True,random_state=26)
knn = KNeighborsRegressor()
pipe = Pipeline([('SS',ss),('KNN', knn)])
params = {'KNN__n_neighbors' : [1,2,3,4,5,6], 'SS': [ss, mm, None]}


In [13]:
gcv = GridSearchCV(estimator = pipe, param_grid = params, cv = kfold, n_jobs = -1, scoring = 'neg_root_mean_squared_error')
gcv.fit(X, y)
gcv.best_params_, gcv.best_score_

({'KNN__n_neighbors': 3, 'SS': StandardScaler()}, -8.710276753396682)

In [19]:
tst = pd.read_csv("testConcreteMissing.csv")
tst.fillna(0, inplace = True)

In [16]:
knn = KNeighborsRegressor(n_neighbors = 3)
bm = Pipeline([('SS', ss), ('KNN', knn)])
bm.fit(X, y)

Pipeline(steps=[('SS', StandardScaler()),
                ('KNN', KNeighborsRegressor(n_neighbors=3))])

In [20]:
bm.predict(tst)

array([57.69333333, 56.21      , 27.00333333, 34.37      , 41.87      ,
       46.27      , 41.74      , 52.36666667, 40.63      , 54.84666667,
       24.09333333, 37.88666667, 52.84      , 43.24666667])

In [21]:
gcv.predict(tst)

array([57.69333333, 56.21      , 27.00333333, 34.37      , 41.87      ,
       46.27      , 41.74      , 52.36666667, 40.63      , 54.84666667,
       24.09333333, 37.88666667, 52.84      , 43.24666667])

In [23]:
gcv.best_estimator_

Pipeline(steps=[('SS', StandardScaler()),
                ('KNN', KNeighborsRegressor(n_neighbors=3))])